# Batch Scoring — Retail Demand Forecasting

Scores all store-product combinations with the champion **registered model**
to produce demand predictions and inventory risk rankings.

**Reads from:** `gold_ml_features`, `gold_ml_scoring_meta`, MLflow registry

**Writes to:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
import json
import mlflow
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round
)

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load champion registered model + scoring metadata
meta = json.loads(spark.read.table('gold_ml_scoring_meta').collect()[0]['meta_json'])
champion_name = meta['champion_model']
feature_cols = meta['feature_cols']
numeric_features = meta['numeric_features']
cat_cols = meta['cat_cols']
category_maps = meta['category_maps']

model = mlflow.pyfunc.load_model(f'models:/{champion_name}/latest')
features_df = spark.read.table('gold_ml_features')
print(f'Loaded registered model: {champion_name}')
print(f'Scoring {features_df.count():,} feature rows')

In [ ]:
# Prepare features exactly as in training: named float64 pandas columns
pdf = features_df.toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)
for c in cat_cols:
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

X_all = pdf[feature_cols].astype('float64')
pdf['prediction'] = model.predict(X_all)
print(f'Scored {len(pdf):,} rows with {champion_name}')

In [ ]:
# Back to Spark for Delta writes — schema identical to the previous version
scored = spark.createDataFrame(pdf)

predictions = (
    scored
    .withColumn('predicted_demand', spark_round(col('prediction'), 1))
    .withColumn('demand_vs_actual', spark_round(col('prediction') - col('daily_quantity'), 1))
    .withColumn('demand_signal',
        when(col('prediction') > col('daily_quantity') * 1.3, 'high_demand')
        .when(col('prediction') < col('daily_quantity') * 0.7, 'low_demand')
        .otherwise('stable')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'store_id', 'product_id', 'txn_date',
        'category', 'subcategory', 'region', 'store_format',
        'daily_quantity', 'predicted_demand', 'demand_vs_actual',
        'daily_revenue', 'avg_price', 'avg_discount',
        'demand_signal',
        'scored_at'
    )
)

predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
print(f'\nDemand signal distribution:')
predictions.groupBy('demand_signal').count().orderBy('count', ascending=False).show()

In [ ]:
# Summary: per product-category demand outlook
summary = (
    predictions
    .groupBy('category', 'region')
    .agg(
        spark_round(avg('predicted_demand'), 1).alias('avg_predicted_demand'),
        spark_round(avg('daily_quantity'), 1).alias('avg_actual_demand'),
        spark_round(avg('demand_vs_actual'), 1).alias('avg_demand_gap'),
        spark_round(spark_sum('daily_revenue'), 0).alias('total_revenue'),
        count('*').alias('observations'),
        spark_sum(when(col('demand_signal') == 'high_demand', 1).otherwise(0)).alias('high_demand_days'),
        spark_sum(when(col('demand_signal') == 'low_demand', 1).otherwise(0)).alias('low_demand_days'),
    )
    .withColumn('demand_trend',
        when(col('avg_demand_gap') > 2, 'growing')
        .when(col('avg_demand_gap') < -2, 'declining')
        .otherwise('stable')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('total_revenue').desc())
)

summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'\nDemand summary: {summary.count()} category-region combos')
summary.show(10, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('\nAll Gold ML tables optimized')

print('\n=== Scoring Summary ===')
print(f'Champion model: {champion_name}')
print(f'Total predictions: {predictions.count():,}')
print(f'Categories scored: {summary.count()}')
print('\nGold tables created:')
print('  - gold_ml_features')
print('  - gold_ml_model_metrics')
print('  - gold_ml_feature_importance')
print('  - gold_ml_predictions')
print('  - gold_ml_summary')
print('  - gold_ml_scoring_meta')
print('\nRegistered models: retail_demand_lgbm, retail_demand_rf')